# 09 Survival Analysis: After Onset, Whose Prognosis Is Worse?

Of the 121 infected residents at Songbai Nursing Home, 19 died.
The attending physician asks: "Which people had a higher risk of death? Can we quantify it?"

**Workflow**: build the analysis dataset → KM curve for everyone → grouped comparison → log-rank test → Cox regression → HR forest plot → **verify the PH assumption**

> 💡 Read this alongside `09_survival.md`, which has plain-language explanations, SVG diagrams, and result-interpretation guides; this notebook is the hands-on "read the code + read the output" version.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — Build the Analysis Dataset

This cell does three things:
1. Reads in the line-list data
2. Marks who is an "event" (event=1 means death) and who is "censored" (event=0 means still alive)
3. Computes each person's `time_to_event` (in days)

> **Small key point**: for survivors, `time_to_event` uses "the last onset date + 14 days" as the observation cutoff, meaning "we watched at least this long and saw no death." This is **not** 0, and it is **not** a missing value.

In [ ]:
# --- Step 1: Build the survival analysis dataset ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (avoids Chinese labels showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Restrict to infected residents
cases = df[df["infected"] == 1].copy()

# Event indicator: 1=died, 0=survived (right-censored)
cases["event"] = (cases["outcome"] == "dead").astype(int)

# Survival time
# Those who died: time = death_date - symptom_onset_date
# Survivors: time = investigation_end - symptom_onset_date (censored)
investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

print(f"Infected residents: {len(cases)}")
print(f"Deaths: {cases['event'].sum()}")
print(f"Survived (censored): {(cases['event'] == 0).sum()}")
print(f"\nInvestigation end date: {investigation_end.date()}")
print(f"\nSurvival time of those who died (days):")
died = cases[cases["event"] == 1]
print(f"  mean = {died['time_to_event'].mean():.1f}")
print(f"  median = {died['time_to_event'].median():.1f}")
print(f"  range = {died['time_to_event'].min()} – {died['time_to_event'].max()} days")

**Interpreting the result**: the output should show 121 infected residents, with 19 deaths and 102 censored. The survival-time range of those who died tells you "the fewest days until someone passed" and "the latest day someone passed"—this is very helpful for understanding the shape of the KM curve later.

## Step 2 — Kaplan-Meier Survival Curve for Everyone

KM is simply **watching how the proportion "still alive" declines each day**. Each time someone dies → the curve steps down; each time someone is censored → a tick is placed on the curve.

> Reading this alongside the [KM curve breakdown figure] in the chapter will quickly get you a handle on the four elements: steps, ticks, median, and CI band.

In [ ]:
# --- Step 2: Kaplan-Meier survival curve for everyone ---
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
kmf.fit(cases["time_to_event"], event_observed=cases["event"],
        label="All infected residents")

fig, ax = plt.subplots(figsize=(8, 5))
kmf.plot_survival_function(ax=ax)
ax.set_title("Kaplan-Meier Survival Curve (all infected residents)")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Median survival time
median_surv = kmf.median_survival_time_
print(f"Median survival time: {median_surv}")
print("→ If the median shows inf, it means over 50% of people survived the observation period (this is good news)")

**How to read this curve:**

1. **Step down** → a resident died on that day
2. **Small vertical tick** → a case still alive (censored)
3. **The day the curve crosses y=0.5** → the median survival time; if it never crosses → `inf` (more than half the people did not die during the observation period ≈ **CFR under 50%**)
4. **The shaded band** = the 95% confidence interval; the wider the tail, the fewer the samples at that time point and the higher the uncertainty

> In this case CFR ≈ 15.7%, far below 50%, so the median survival time is `inf`—this is **good news**, not a bug.

## Step 3 — Survival Curves by Severity Group

Split cases by `clinical_severity` into three groups (mild / moderate / severe) and overlay three KM curves.

**Three perspectives for reading the figure**:
- **The earlier they separate** → the more immediate the factor's effect
- **The larger the gap** → the stronger the effect
- **Curves cross** → the PH assumption may be violated (Step 7 will verify)

In [ ]:
# --- Step 3: Survival curves by severity group ---
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

fig, ax = plt.subplots(figsize=(8, 5))

for sev in severity_levels:
    mask = cases["clinical_severity"] == sev
    sub = cases[mask]
    if len(sub) == 0:
        continue
    kmf_sev = KaplanMeierFitter()
    kmf_sev.fit(sub["time_to_event"], event_observed=sub["event"],
                label=f"{sev} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf_sev.plot_survival_function(ax=ax, color=colors[sev])

ax.set_title("Survival curves (grouped by severity)")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print("→ Observe: is the severe group's survival curve clearly below mild/moderate?")
print("→ The earlier the curves separate and the larger the gap = the stronger severity's effect on survival")

**Interpreting the result**:

- We expect the `severe` group's curve to drop the earliest and fastest—because severely ill patients have the highest short-term risk of death.
- If the `severe` curve is clearly below `mild` and `moderate`, severity is an important prognostic indicator.
- If you observe `severe` and `moderate` **crossing**, make a special note—this hints that the effect of severity may **change over time** (violating the PH assumption).

## Step 4 — Log-rank Test

Step 3 is **visual judgment**; Step 4 is **statistical inference**:

- **H₀**: the two groups' survival curves have the same shape
- **H₁**: the hazards differ at at least one time point
- **p < 0.05** → reject H₀; the difference between the two groups is significant

⚠️ The log-rank test **only tells you "whether there's a difference"**, **not how big it is**—to quantify the effect (HR) you have to wait for the Cox model in Step 5.

In [ ]:
# --- Step 4: Log-rank test ---
from lifelines.statistics import logrank_test

# Compare severe vs non-severe
severe = cases[cases["clinical_severity"] == "severe"]
non_severe = cases[cases["clinical_severity"].isin(["mild", "moderate"])]

result = logrank_test(
    severe["time_to_event"], non_severe["time_to_event"],
    event_observed_A=severe["event"],
    event_observed_B=non_severe["event"],
)

print("=== Log-rank test: severe vs non-severe ===")
print(f"  test statistic = {result.test_statistic:.3f}")
print(f"  p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("  → p < 0.05: the two survival curves differ significantly")
else:
    print("  → p ≥ 0.05: cannot reject the null hypothesis that the two survival curves are the same")

# Also compare COPD vs no COPD
copd_yes = cases[cases["comorbidity_copd"] == 1]
copd_no = cases[cases["comorbidity_copd"] == 0]

result_copd = logrank_test(
    copd_yes["time_to_event"], copd_no["time_to_event"],
    event_observed_A=copd_yes["event"],
    event_observed_B=copd_no["event"],
)

print(f"\n=== Log-rank test: COPD vs no COPD ===")
print(f"  test statistic = {result_copd.test_statistic:.3f}")
print(f"  p-value = {result_copd.p_value:.4f}")

**Interpreting the result**:

- `test statistic` ≈ χ²(1) distribution; the larger the value → the more different the two groups
- `p_value` < 0.05 → treated as "statistically significant"; ≥ 0.05 → insufficient evidence

> Think about it: the COPD group may not have many people (about 28% have COPD in this case). If p is large, it could be **truly no difference**, or it could be **too few samples, underpowered**—don't directly read "not significant" as "no association."

## Step 5 — Cox Proportional Hazards Regression

Cox adjusts for multiple variables at once and tells you **each variable's independent HR**.

**Rule of thumb for reading `print_summary()`**: "Just look at `exp(coef)` and its CI."

| Column | Meaning |
|------|------|
| `exp(coef)` | **HR** (1.5 = risk is 1.5 times faster) |
| `exp(coef) lower/upper 95%` | The HR's 95% CI; **crosses 1 → not significant** |
| `p` | p-value (< 0.05 significant) |
| `Concordance` | The model's overall ranking ability (0.5 random, >0.7 acceptable, >0.8 good) |

In [ ]:
# --- Step 5: Cox proportional hazards regression ---
from lifelines import CoxPHFitter

# Build the Cox regression dataset
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "comorbidity_copd", "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "immunosuppressed",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

# Fit the model
cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox proportional hazards regression results ===")
cph.print_summary()

# Concise HR table
print("\n=== Hazard Ratio summary ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n→ HR > 1 means a higher risk of death (risk factor)")
print("→ HR < 1 means a lower risk of death (protective factor)")
print("→ If the 95% CI contains 1, it is not significant")

**⚠️ Sample size warning (events per variable, EPV)**

- This case has only **19 death events**, yet we put in **7 variables** (age, is_male, 4 comorbidities, immunosuppressed)
- Epidemiological rule of thumb: **at least 10 events per variable** (Peduzzi 1995)
- `19 / 7 ≈ 2.7`, far below 10 → **this chapter is a teaching demonstration**; in practice a model like this easily overfits
- This also explains why many variables "look non-significant"—it's not that they truly don't matter, but that the sample doesn't have enough power to support this many variables

**Practical advice**: do variable selection first (like the Modified Poisson crude RR in Ch06), keeping only the 1-2 most crucial adjustment factors.

## Step 6 — HR Forest Plot

`lifelines`'s `cph.plot()` draws **log(HR)**—so the vertical dashed line at x=0 represents **HR = 1**.

**Three steps to read it**:
1. Which side of 0 is the dot on? → risk vs protective
2. Does the error bar cross 0? → crossing means not significant
3. How long is the error bar? → the longer it is, the wider the CI and the higher the uncertainty

In [ ]:
# --- Step 6: HR forest plot ---
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression — Hazard Ratio (log scale)")
plt.tight_layout()
plt.show()

print("→ The dots in the forest plot = log(HR); the error bars = 95% CI")
print("→ Dot to the right of the dashed line = HR > 1 (risk factor)")
print("→ Error bar crossing the dashed line = not significant")

**Interpreting the result**:

- **Dot + line "entirely to the right of 0"** → the variable is a statistically significant risk factor
- **Dot + line "entirely to the left of 0"** → a statistically significant protective factor
- **Line crosses x=0** → the 95% CI crosses 1 → not significant
- **Very long line** → few events / high variance, unstable estimate—interpret with caution in practice

> In this case events are few (19 events), so the lines for many variables will be quite long—this is a limitation of the data size, not a fault of Cox.

## Step 7 — Verify the PH Assumption (brand new)

Cox's **proportional hazards (PH) assumption**: the hazard ratio between the two groups stays constant throughout the entire follow-up period.
If it's violated, the HR computed in Step 5 becomes an "average effect," masking the truth that changes over time.

Verify it in one line with `cph.check_assumptions()`:

- Run a **Schoenfeld residuals** test for each variable
- Print each variable's p-value + recommendations
- `No violation detected` → ✓ pass; a variable with `p < 0.05` → ✗ violation

> `show_plots=False` makes it print text conclusions only (avoiding cluttering the notebook with lots of subplots).

In [ ]:
# --- Step 7: Verify the PH assumption ---
print("=== Checking the proportional hazards (PH) assumption ===\n")

# show_plots=False: print text conclusions only, avoiding extra subplots
results = cph.check_assumptions(cox_df, show_plots=False, advice=True)

print("\n→ No violation detected for any variable → the Cox results are trustworthy")
print("→ If a violation is detected: use strata, a time-varying coefficient, or switch to an AFT model")

**Interpreting the result**:

| Output | Meaning | Next step |
|------|------|--------|
| `proportional_hazard_test ... No violation detected` | ✓ Passes the test | The Cox results are trustworthy |
| A variable shows `p < 0.05` | ✗ That variable violates PH | Consider a remedy (see below) |

**Remedies when PH is violated (simplest to most advanced)**:
1. **Stratify (strata)**: `cph.fit(..., strata=["the violating variable"])` —— allows that variable's baseline hazard to vary freely
2. **Time-varying coefficient**: let the variable's effect change over time
3. **Split into time periods**: e.g., run separate Cox models for "the first two weeks" and "the last two weeks"
4. **Switch to an AFT (Accelerated Failure Time) model**: bypass the PH assumption entirely

⚠️ **A note when events are few**: this case has only 19 events, so `check_assumptions()` is underpowered—even if no violation is detected, it doesn't mean PH definitely holds. **Always pair it with the grouped KM visual check in Step 3** (do the curves cross?).

## Supplement — COPD Grouped Kaplan-Meier

An extra demonstration: use `comorbidity_copd` to draw KM curves for two groups, echoing the log-rank result from Step 4.

In [ ]:
# --- Supplement: COPD grouped Kaplan-Meier ---
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("COPD", cases["comorbidity_copd"] == 1),
                     ("No COPD", cases["comorbidity_copd"] == 0)]:
    sub = cases[mask]
    kmf_sub = KaplanMeierFitter()
    kmf_sub.fit(sub["time_to_event"], event_observed=sub["event"],
                label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf_sub.plot_survival_function(ax=ax)

ax.set_title("Survival curves: COPD vs No COPD")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print(f"\nCOPD Log-rank p-value = {result_copd.p_value:.4f}")

## Summary

| Step | Skill learned |
|------|------------|
| Step 1 Build the dataset | Compute the `time_to_event` and `event` indicators, handling censoring correctly |
| Step 2 KM curve | `KaplanMeierFitter` survival curve for everyone; read the steps, ticks, median, and CI band |
| Step 3 Grouped KM | Grouped comparison: look at the **point of separation**, the **gap**, and **whether they cross** |
| Step 4 Log-rank | `logrank_test()` compares differences between two groups; interpreting H₀/H₁ and the p-value |
| Step 5 Cox regression | `CoxPHFitter` multi-factor analysis; read every column of `print_summary()` |
| Step 6 Forest plot | `cph.plot()` visualizes the HR; three steps to read the figure |
| **Step 7 PH diagnosis** | `cph.check_assumptions()` verifies whether the Cox results are trustworthy |

**Conclusion**: survival analysis is more precise than a plain case fatality rate (CFR)—it considers both "whether someone died" and "how fast they died."
Cox regression can adjust for multiple factors at once and find the independent prognostic risk factors. But don't forget to verify the PH assumption, and don't ignore the events-per-variable warning.

In the next chapter (Ch10), we'll try to train a machine learning model using all the features → predicting infection and severe illness.